# Silver-to-Gold Data Transformation and Feature Engineering Notebook
### Airline Delay Analytics & Machine Learning Pipeline using PySpark on Amazon EMR

## Overview

This notebook implements the **Silver-to-Gold transformation layer** of the Airline Delay Analytics project. It converts the cleaned Silver dataset into an analytics-ready Gold layer by applying feature engineering, dimensional modelling, and machine learning dataset preparation.

The pipeline follows a **Star Schema architecture**, enabling efficient business intelligence reporting, dashboard development, and predictive machine learning.

## Objectives

- Load the cleaned Silver dataset from Amazon S3.
- Create business-relevant engineered features.
- Generate business and surrogate keys.
- Build the Gold Base dataset.
- Create Fact and Dimension tables following Star Schema principles.
- Prepare a Machine Learning dataset for arrival delay prediction.
- Validate the generated datasets before writing them to Amazon S3.
- Store all Gold datasets in optimized Parquet format.

## Gold Layer Outputs

The notebook produces the following six Gold datasets:

1. FACT_FLIGHTS
2. DIM_AIRLINE
3. DIM_AIRPORT
4. DIM_DATE
5. DIM_ROUTE
6. ML_DATASET

## 1. Imports, Spark settings, and Gold paths

In [1]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark import StorageLevel

# Runtime-safe SQL settings for the 1-primary + 2-core m5.xlarge cluster.
spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.conf.set("spark.sql.parquet.mergeSchema", "false")
spark.conf.set("spark.sql.shuffle.partitions", "64")
spark.conf.set("spark.sql.broadcastTimeout", "900")
spark.conf.set(
    "spark.sql.files.maxPartitionBytes",
    str(128 * 1024 * 1024)
)

# AQE is available only in newer Spark versions.
# The configuration is enabled only when supported.
spark_major_minor = tuple(
    int(part) for part in spark.version.split(".")[:2]
)

if spark_major_minor >= (3, 0):
    spark.conf.set("spark.sql.adaptive.enabled", "true")
    spark.conf.set(
        "spark.sql.adaptive.coalescePartitions.enabled",
        "true"
    )
    spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

INPUT_PATH = "s3://airline-dataset-2020-2025/Silver/Flight_Data_2020_2025/"
GOLD_BASE_PATH = "s3://airline-dataset-2020-2025/Gold/"

FACT_FLIGHTS_PATH = GOLD_BASE_PATH + "FACT_FLIGHTS/"
DIM_AIRLINE_PATH = GOLD_BASE_PATH + "DIM_AIRLINE/"
DIM_AIRPORT_PATH = GOLD_BASE_PATH + "DIM_AIRPORT/"
DIM_DATE_PATH = GOLD_BASE_PATH + "DIM_DATE/"
DIM_ROUTE_PATH = GOLD_BASE_PATH + "DIM_ROUTE/"
ML_DATASET_PATH = GOLD_BASE_PATH + "ML_DATASET/"

WRITE_OUTPUT = True
OUTPUT_MODE = "overwrite"

TRAIN_END_DATE = "2023-12-31"
VALIDATION_YEAR = 2024
TEST_YEAR = 2025

print("Spark version:", spark.version)
print("Default parallelism:", spark.sparkContext.defaultParallelism)
print(
    "Shuffle partitions:",
    spark.conf.get("spark.sql.shuffle.partitions")
)
print(
    "Broadcast timeout:",
    spark.conf.get("spark.sql.broadcastTimeout")
)
print("Silver input:", INPUT_PATH)
print("Gold base:", GOLD_BASE_PATH)
print("Training history ends:", TRAIN_END_DATE)


Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,Current session?
0,application_1784815217303_0001,pyspark,idle,Link,Link,✔


SparkSession available as 'spark'.
('Spark version:', u'2.4.0')
('Default parallelism:', 2)
('Shuffle partitions:', u'64')
('Broadcast timeout:', u'900')
('Silver input:', 's3://airline-dataset-2020-2025/Silver/Flight_Data_2020_2025/')
('Gold base:', 's3://airline-dataset-2020-2025/Gold/')
('Training history ends:', '2023-12-31')

## 2. Load the Silver-layer Parquet dataset

In [2]:
silver_df = spark.read.parquet(INPUT_PATH)
print("Source columns:", len(silver_df.columns))
silver_df.printSchema()

('Source columns:', 120)
root
 |-- Quarter: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- DayofMonth: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- FlightDate: timestamp (nullable = true)
 |-- Marketing_Airline_Network: string (nullable = true)
 |-- Operated_or_Branded_Code_Share_Partners: string (nullable = true)
 |-- DOT_ID_Marketing_Airline: integer (nullable = true)
 |-- IATA_Code_Marketing_Airline: string (nullable = true)
 |-- Flight_Number_Marketing_Airline: integer (nullable = true)
 |-- Originally_Scheduled_Code_Share_Airline: string (nullable = true)
 |-- DOT_ID_Originally_Scheduled_Code_Share_Airline: integer (nullable = true)
 |-- IATA_Code_Originally_Scheduled_Code_Share_Airline: string (nullable = true)
 |-- Flight_Num_Originally_Scheduled_Code_Share_Airline: integer (nullable = true)
 |-- Operating_Airline: string (nullable = true)
 |-- DOT_ID_Operating_Airline: integer (nullable = true)
 |-- IATA_Code_Operating_Airline: 

## 3. Select required Silver columns

The agreed Gold process excludes `CancellationCode` and `Tail_Number`.

`CRSElapsedTime` is used when available because scheduled duration is known before departure and is suitable for arrival-delay modelling.


In [3]:
required_columns = [
    "Year", "Quarter", "Month", "DayofMonth", "DayOfWeek", "FlightDate",

    "Marketing_Airline_Network",
    "Operating_Airline",
    "Operated_or_Branded_Code_Share_Partners",
    "Flight_Number_Marketing_Airline",

    "Origin", "OriginCityName", "OriginState", "OriginStateName",
    "Dest", "DestCityName", "DestState", "DestStateName",

    "CRSDepTime", "CRSArrTime",

    "DepDelay", "ArrDelay", "DepDel15", "ArrDel15",

    "CarrierDelay", "WeatherDelay", "NASDelay",
    "SecurityDelay", "LateAircraftDelay",

    "Cancelled", "Diverted",

    "Distance", "AirTime", "TaxiOut", "TaxiIn"
]

optional_columns = [
    "CRSElapsedTime"
]

missing = sorted(set(required_columns) - set(silver_df.columns))
if missing:
    raise ValueError("Missing required Silver columns: " + ", ".join(missing))

selected_columns = required_columns + [
    c for c in optional_columns if c in silver_df.columns
]

base_df = silver_df.select(*selected_columns)

print("Selected columns:", len(base_df.columns))
print("Optional columns found:", [
    c for c in optional_columns if c in base_df.columns
])


('Selected columns:', 36)
('Optional columns found:', ['CRSElapsedTime'])

## 4. Datatype handling and standardisation

- FlightDate → Date
- Calendar fields → Integer
- Delay, distance, air-time and taxi-time measures → Integer
- Binary indicators → Integer 0/1 flags
- State, airport and airline fields → trimmed and standardized
- Original operational nulls are preserved

Binary flags remain integers because they are convenient for aggregation and Spark ML pipelines.


In [4]:
calendar_cols = ["Year", "Quarter", "Month", "DayofMonth", "DayOfWeek"]

measure_cols = [
    "DepDelay", "ArrDelay",
    "CarrierDelay", "WeatherDelay", "NASDelay",
    "SecurityDelay", "LateAircraftDelay",
    "Distance", "AirTime", "TaxiOut", "TaxiIn"
]

if "CRSElapsedTime" in base_df.columns:
    measure_cols.append("CRSElapsedTime")

binary_cols = ["Cancelled", "Diverted", "DepDel15", "ArrDel15"]

typed_df = base_df.withColumn("FlightDate", F.to_date("FlightDate"))

for c in calendar_cols:
    typed_df = typed_df.withColumn(c, F.col(c).cast("int"))

for c in measure_cols:
    typed_df = typed_df.withColumn(c, F.round(F.col(c)).cast("int"))

for c in binary_cols:
    typed_df = typed_df.withColumn(
        c,
        F.when(F.col(c).isNull(), F.lit(None).cast("int"))
         .otherwise(F.col(c).cast("int"))
    )

typed_df = (
    typed_df
    .withColumn("CRSDepTime", F.col("CRSDepTime").cast("int"))
    .withColumn("CRSArrTime", F.col("CRSArrTime").cast("int"))
    .withColumn(
        "Flight_Number_Marketing_Airline",
        F.col("Flight_Number_Marketing_Airline").cast("int")
    )
    .withColumn("Origin", F.upper(F.trim(F.col("Origin"))))
    .withColumn("Dest", F.upper(F.trim(F.col("Dest"))))
    .withColumn("OriginState", F.upper(F.trim(F.col("OriginState"))))
    .withColumn("DestState", F.upper(F.trim(F.col("DestState"))))
    .withColumn(
        "Marketing_Airline_Network",
        F.upper(F.trim(F.col("Marketing_Airline_Network")))
    )
    .withColumn(
        "Operating_Airline",
        F.upper(F.trim(F.col("Operating_Airline")))
    )
    .withColumn(
        "Operated_or_Branded_Code_Share_Partners",
        F.upper(
            F.trim(F.col("Operated_or_Branded_Code_Share_Partners"))
        )
    )
)

typed_df.printSchema()


root
 |-- Year: integer (nullable = true)
 |-- Quarter: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- DayofMonth: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- FlightDate: date (nullable = true)
 |-- Marketing_Airline_Network: string (nullable = true)
 |-- Operating_Airline: string (nullable = true)
 |-- Operated_or_Branded_Code_Share_Partners: string (nullable = true)
 |-- Flight_Number_Marketing_Airline: integer (nullable = true)
 |-- Origin: string (nullable = true)
 |-- OriginCityName: string (nullable = true)
 |-- OriginState: string (nullable = true)
 |-- OriginStateName: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- DestCityName: string (nullable = true)
 |-- DestState: string (nullable = true)
 |-- DestStateName: string (nullable = true)
 |-- CRSDepTime: integer (nullable = true)
 |-- CRSArrTime: integer (nullable = true)
 |-- DepDelay: integer (nullable = true)
 |-- ArrDelay: integer (nullable = true)
 |-- D

## 5. Convert scheduled HHMM values into usable time features

The notebook creates departure/arrival hour, minute and `HH:MM` display values.

It also creates `ScheduledElapsedTimeMinutes`:

- Uses `CRSElapsedTime` when available.
- Otherwise derives the scheduled interval from departure and arrival HHMM values, including overnight flights.


In [6]:
def add_hhmm_features(df, source_col, prefix):
    normalized = F.when(F.col(source_col) == 2400, 0).otherwise(
        F.col(source_col)
    )

    hour = F.floor(normalized / 100).cast("int")
    minute = (normalized % 100).cast("int")

    valid = (
        normalized.isNotNull()
        & hour.between(0, 23)
        & minute.between(0, 59)
    )

    return (
        df
        .withColumn(
            "{}Hour".format(prefix),
            F.when(valid, hour).otherwise(F.lit(None).cast("int"))
        )
        .withColumn(
            "{}Minute".format(prefix),
            F.when(valid, minute).otherwise(F.lit(None).cast("int"))
        )
        .withColumn(
            "{}TimeHHMM".format(prefix),
            F.when(valid, F.format_string("%02d:%02d", hour, minute))
        )
    )

feature_df = add_hhmm_features(typed_df, "CRSDepTime", "Departure")
feature_df = add_hhmm_features(feature_df, "CRSArrTime", "Arrival")

scheduled_difference = (
    (
        F.col("ArrivalHour") * 60
        + F.col("ArrivalMinute")
        - F.col("DepartureHour") * 60
        - F.col("DepartureMinute")
        + 1440
    ) % 1440
).cast("int")

if "CRSElapsedTime" in feature_df.columns:
    feature_df = feature_df.withColumn(
        "ScheduledElapsedTimeMinutes",
        F.coalesce(F.col("CRSElapsedTime"), scheduled_difference)
    )
else:
    feature_df = feature_df.withColumn(
        "ScheduledElapsedTimeMinutes",
        scheduled_difference
    )


## 6. Calendar, weekend, season-indicator and year-month features

In [7]:
feature_df = (
    feature_df
    .withColumn("WeekendIndicator", F.when(F.col("DayOfWeek").isin(6, 7), 1).otherwise(0))
    .withColumn(
        "SeasonIndicator",
        F.when(F.col("Month").isin(12, 1, 2), "Winter")
         .when(F.col("Month").isin(3, 4, 5), "Spring")
         .when(F.col("Month").isin(6, 7, 8), "Summer")
         .when(F.col("Month").isin(9, 10, 11), "Fall")
         .otherwise("Unknown")
    )
    .withColumn(
        "YearMonth",
        F.concat_ws("-", F.col("Year"), F.lpad(F.col("Month"), 2, "0"))
    )
    .withColumn(
        "FlightYearMonth",
        F.to_date(F.concat_ws("-", F.col("Year"), F.lpad(F.col("Month"), 2, "0"), F.lit("01")))
    )
)

## 7. Departure/arrival period and peak-hour indicators

Periods:
- Morning: 05:00–11:59
- Afternoon: 12:00–16:59
- Evening: 17:00–21:59
- Night: 22:00–04:59

Peak departure windows are defined as 06:00–09:59 and 16:00–19:59.

In [8]:
feature_df = (
    feature_df
    .withColumn(
        "DeparturePeriod",
        F.when(F.col("DepartureHour").between(5, 11), "Morning")
         .when(F.col("DepartureHour").between(12, 16), "Afternoon")
         .when(F.col("DepartureHour").between(17, 21), "Evening")
         .when(F.col("DepartureHour").isNotNull(), "Night")
         .otherwise("Unknown")
    )
    .withColumn(
        "ArrivalPeriod",
        F.when(F.col("ArrivalHour").between(5, 11), "Morning")
         .when(F.col("ArrivalHour").between(12, 16), "Afternoon")
         .when(F.col("ArrivalHour").between(17, 21), "Evening")
         .when(F.col("ArrivalHour").isNotNull(), "Night")
         .otherwise("Unknown")
    )
    .withColumn(
        "PeakHourIndicator",
        F.when(
            F.col("DepartureHour").between(6, 9) |
            F.col("DepartureHour").between(16, 19), 1
        ).otherwise(0)
    )
)

## 8. Route, state-pair and composite business-key features

- Route: directional airport pair, for example ATL-JFK
- StatePair: directional state pair, for example GA-NY
- IntraStateRouteFlag: 1 when origin and destination are in the same state
- FlightKey: composite scheduled-flight business key

A general domestic-flight flag is not created because the source is already a domestic-flight dataset.


In [9]:
feature_df = (
    feature_df
    .withColumn(
        "Route",
        F.concat_ws("-", F.col("Origin"), F.col("Dest"))
    )
    .withColumn(
        "StatePair",
        F.concat_ws("-", F.col("OriginState"), F.col("DestState"))
    )
    .withColumn(
        "IntraStateRouteFlag",
        F.when(
            F.col("OriginState").isNotNull()
            & F.col("DestState").isNotNull()
            & (F.col("OriginState") == F.col("DestState")),
            1
        ).otherwise(0)
    )
    .withColumn(
        "FlightKey",
        F.concat_ws(
            "|",
            F.date_format("FlightDate", "yyyy-MM-dd"),
            F.coalesce(
                F.col("Marketing_Airline_Network"),
                F.lit("UNK")
            ),
            F.coalesce(
                F.col("Flight_Number_Marketing_Airline").cast("string"),
                F.lit("UNK")
            ),
            F.coalesce(F.col("Origin"), F.lit("UNK")),
            F.coalesce(F.col("Dest"), F.lit("UNK")),
            F.lpad(
                F.coalesce(
                    F.col("CRSDepTime").cast("string"),
                    F.lit("0")
                ),
                4,
                "0"
            )
        )
    )
)


## 9. Flight time, distance, categories and availability flags

The original nulls are preserved.

- TotalFlightTimeMinutes is calculated only when AirTime, TaxiOut, and TaxiIn are all available.
- FlightTimeCompleteFlag shows whether the total can be calculated.
- OperationalTimeMissingReason explains common structural missingness.
- AverageFlightSpeedMph is calculated only when distance and positive air time are available.
- Missing categories are labelled Not Available; null values are not silently converted to zero.


In [10]:
feature_df = (
    feature_df
    .withColumn(
        "FlightTimeCompleteFlag",
        F.when(
            F.col("AirTime").isNotNull()
            & F.col("TaxiOut").isNotNull()
            & F.col("TaxiIn").isNotNull(),
            1
        ).otherwise(0)
    )
    .withColumn(
        "TotalFlightTimeMinutes",
        F.when(
            F.col("FlightTimeCompleteFlag") == 1,
            F.col("AirTime") + F.col("TaxiOut") + F.col("TaxiIn")
        ).otherwise(F.lit(None).cast("int"))
    )
    .withColumn(
        "OperationalTimeMissingReason",
        F.when(F.col("FlightTimeCompleteFlag") == 1, "Available")
         .when(F.col("Cancelled") == 1, "Cancelled")
         .when(F.col("Diverted") == 1, "Diverted")
         .otherwise("Other Missing")
    )
    .withColumn(
        "SpeedAvailableFlag",
        F.when(
            F.col("Distance").isNotNull()
            & F.col("AirTime").isNotNull()
            & (F.col("AirTime") > 0),
            1
        ).otherwise(0)
    )
    .withColumn(
        "AverageFlightSpeedMph",
        F.when(
            F.col("SpeedAvailableFlag") == 1,
            F.round(F.col("Distance") / F.col("AirTime") * 60, 2)
        )
    )
    .withColumn(
        "FlightDurationCategory",
        F.when(
            F.col("TotalFlightTimeMinutes").isNull(),
            "Not Available"
        )
        .when(F.col("TotalFlightTimeMinutes") < 120, "Short")
        .when(F.col("TotalFlightTimeMinutes") < 240, "Medium")
        .otherwise("Long")
    )
    .withColumn(
        "DistanceCategory",
        F.when(F.col("Distance").isNull(), "Not Available")
         .when(F.col("Distance") < 500, "Short Haul")
         .when(F.col("Distance") < 1500, "Medium Haul")
         .otherwise("Long Haul")
    )
)


## 10. Delay availability, null reasons, performance type and delay category

Null delay values are handled explicitly before numerical comparisons.

- A cancelled or diverted flight may legitimately have no normal departure/arrival delay.
- HasDepDelay and HasArrDelay identify availability.
- DepDelayMissingReason and ArrDelayMissingReason distinguish structural nulls.
- ArrDelayNullExceptionFlag identifies missing arrival delay on a flight marked neither cancelled nor diverted.


In [11]:
feature_df = (
    feature_df
    .withColumn(
        "HasDepDelay",
        F.when(F.col("DepDelay").isNotNull(), 1).otherwise(0)
    )
    .withColumn(
        "HasArrDelay",
        F.when(F.col("ArrDelay").isNotNull(), 1).otherwise(0)
    )
    .withColumn(
        "DepDelayMissingReason",
        F.when(F.col("DepDelay").isNotNull(), "Available")
         .when(F.col("Cancelled") == 1, "Cancelled")
         .when(F.col("Diverted") == 1, "Diverted")
         .otherwise("Other Missing")
    )
    .withColumn(
        "ArrDelayMissingReason",
        F.when(F.col("ArrDelay").isNotNull(), "Available")
         .when(F.col("Cancelled") == 1, "Cancelled")
         .when(F.col("Diverted") == 1, "Diverted")
         .otherwise("Other Missing")
    )
    .withColumn(
        "ArrDelayNullExceptionFlag",
        F.when(
            F.col("ArrDelay").isNull()
            & (F.coalesce(F.col("Cancelled"), F.lit(0)) == 0)
            & (F.coalesce(F.col("Diverted"), F.lit(0)) == 0),
            1
        ).otherwise(0)
    )
    .withColumn(
        "DeparturePerformanceType",
        F.when(F.col("DepDelay").isNull(), "Not Available")
         .when(F.col("DepDelay") < 0, "Early")
         .when(F.col("DepDelay") == 0, "On Time")
         .otherwise("Delayed")
    )
    .withColumn(
        "ArrivalPerformanceType",
        F.when(F.col("ArrDelay").isNull(), "Not Available")
         .when(F.col("ArrDelay") < 0, "Early")
         .when(F.col("ArrDelay") == 0, "On Time")
         .otherwise("Delayed")
    )
    .withColumn(
        "DelayCategory",
        F.when(F.col("ArrDelay").isNull(), "Not Available")
         .when(F.col("ArrDelay") <= 15, "On Time")
         .when(F.col("ArrDelay") <= 60, "Minor Delay")
         .otherwise("Major Delay")
    )
)


## 11. Delay-cause flags and zero-filled copies

The original cause columns are preserved. Separate filled columns are created so that null meaning is not lost.

In [12]:
cause_cols = ["CarrierDelay", "WeatherDelay", "NASDelay", "SecurityDelay", "LateAircraftDelay"]

for c in cause_cols:
    feature_df = feature_df.withColumn(c + "Filled", F.coalesce(F.col(c), F.lit(0)).cast("int"))

feature_df = feature_df.withColumn(
    "TotalRecordedCauseDelay",
    sum(F.col(c + "Filled") for c in cause_cols)
)

feature_df = feature_df.withColumn(
    "DelayCauseFlag",
    F.when(F.col("TotalRecordedCauseDelay") > 0, 1).otherwise(0)
)

max_cause = F.greatest(*[F.col(c + "Filled") for c in cause_cols])
feature_df = feature_df.withColumn(
    "PrimaryDelayCause",
    F.when(F.col("TotalRecordedCauseDelay") == 0, "None")
     .when(F.col("CarrierDelayFilled") == max_cause, "Carrier")
     .when(F.col("WeatherDelayFilled") == max_cause, "Weather")
     .when(F.col("NASDelayFilled") == max_cause, "NAS")
     .when(F.col("SecurityDelayFilled") == max_cause, "Security")
     .when(F.col("LateAircraftDelayFilled") == max_cause, "Late Aircraft")
     .otherwise("Unknown")
)

## 12. Flight status and completion flag

In [13]:
feature_df = (
    feature_df
    .withColumn(
        "FlightStatus",
        F.when(F.col("Cancelled") == 1, "Cancelled")
         .when(F.col("Diverted") == 1, "Diverted")
         .when(
             (F.col("Cancelled") == 0)
             & (F.col("Diverted") == 0),
             "Completed"
         )
         .otherwise("Unknown")
    )
    .withColumn(
        "CompletedFlightFlag",
        F.when(
            (F.col("Cancelled") == 0)
            & (F.col("Diverted") == 0),
            1
        ).otherwise(0)
    )
)


## 13. Official code-share feature and airline names

Operated_or_Branded_Code_Share_Partners is used as the primary BTS code-share indicator.

CodeshareFlag = 1 when:

- the official partner field contains CODESHARE, or
- the operating airline differs from the marketing airline.

The second condition acts as a defensive fallback for unusual source values.


In [14]:
airline_names = {
    # Major / Network Carriers
    "AA": "American Airlines",
    "AS": "Alaska Airlines",
    "B6": "JetBlue Airways",
    "DL": "Delta Air Lines",
    "F9": "Frontier Airlines",
    "G4": "Allegiant Air",
    "HA": "Hawaiian Airlines",
    "NK": "Spirit Airlines",
    "UA": "United Airlines",
    "WN": "Southwest Airlines",
    # Regional Carriers
    "9E": "Endeavor Air",
    "AX": "Trans States Airlines",
    "C5": "CommutAir",
    "CP": "Compass Airlines",
    "EM": "Empire Airlines",
    "EV": "ExpressJet Airlines",
    "G7": "GoJet Airlines",
    "MQ": "Envoy Air",
    "OH": "PSA Airlines",
    "OO": "SkyWest Airlines",
    "PT": "Piedmont Airlines",
    "QX": "Horizon Air",
    "YV": "Mesa Airlines",
    "YX": "Republic Airways",
    "ZW": "Air Wisconsin",
    # Codeshare Partners
    "AA_CODESHARE": "American Airlines Codeshare Partner",
    "AS_CODESHARE": "Alaska Airlines Codeshare Partner",
    "DL_CODESHARE": "Delta Air Lines Codeshare Partner",
    "HA_CODESHARE": "Hawaiian Airlines Codeshare Partner",
    "UA_CODESHARE": "United Airlines Codeshare Partner",
}

mapping = []
for k, v in airline_names.items():
    mapping += [F.lit(k), F.lit(v)]
airline_map = F.create_map(*mapping)

feature_df = (
    feature_df
    .withColumn(
        "CodeshareFlag",
        F.when(
            F.upper(
                F.coalesce(
                    F.col("Operated_or_Branded_Code_Share_Partners"),
                    F.lit("")
                )
            ).contains("CODESHARE"),
            1
        )
        .when(
            F.col("Marketing_Airline_Network").isNotNull()
            & F.col("Operating_Airline").isNotNull()
            & (
                F.col("Marketing_Airline_Network")
                != F.col("Operating_Airline")
            ),
            1
        )
        .otherwise(0)
    )
    .withColumn(
        "CodesharePartnerLabel",
        F.coalesce(
            F.col("Operated_or_Branded_Code_Share_Partners"),
            F.lit("Not Available")
        )
    )
    .withColumn(
        "MarketingAirlineName",
        F.coalesce(
            airline_map[F.col("Marketing_Airline_Network")],
            F.lit("Unknown Airline")
        )
    )
    .withColumn(
        "OperatingAirlineName",
        F.coalesce(
            airline_map[F.col("Operating_Airline")],
            F.lit("Unknown Airline")
        )
    )
    .withColumn(
        "MarketingAirlineLabel",
        F.concat_ws(
            " - ",
            F.col("Marketing_Airline_Network"),
            F.col("MarketingAirlineName")
        )
    )
    .withColumn(
        "OperatingAirlineLabel",
        F.concat_ws(
            " - ",
            F.col("Operating_Airline"),
            F.col("OperatingAirlineName")
        )
    )
)


## 14. U.S. region grouping

In [15]:
state_regions = {
    "CT":"Northeast","ME":"Northeast","MA":"Northeast","NH":"Northeast","RI":"Northeast","VT":"Northeast","NJ":"Northeast","NY":"Northeast","PA":"Northeast",
    "IL":"Midwest","IN":"Midwest","MI":"Midwest","OH":"Midwest","WI":"Midwest","IA":"Midwest","KS":"Midwest","MN":"Midwest","MO":"Midwest","NE":"Midwest","ND":"Midwest","SD":"Midwest",
    "DE":"South","FL":"South","GA":"South","MD":"South","NC":"South","SC":"South","VA":"South","DC":"South","WV":"South","AL":"South","KY":"South","MS":"South","TN":"South","AR":"South","LA":"South","OK":"South","TX":"South",
    "AZ":"West","CO":"West","ID":"West","MT":"West","NV":"West","NM":"West","UT":"West","WY":"West","AK":"West","CA":"West","HI":"West","OR":"West","WA":"West",
    "PR":"Other","VI":"Other","GU":"Other","TT":"Other"
}

region_mapping=[]
for k,v in state_regions.items():
    region_mapping += [F.lit(k), F.lit(v)]
region_map = F.create_map(*region_mapping)

feature_df = (
    feature_df
    .withColumn("OriginRegion", F.coalesce(region_map[F.col("OriginState")], F.lit("Unknown")))
    .withColumn("DestRegion", F.coalesce(region_map[F.col("DestState")], F.lit("Unknown")))
    .withColumn("RegionRoute", F.concat_ws("-", "OriginRegion", "DestRegion"))
)

## 15. Add deterministic dimension keys

For S3 Parquet, SHA-256 keys are stable across reruns and do not depend on row order.

The Fact table uses separate role-playing airline keys:

- `MarketingAirlineKey`
- `OperatingAirlineKey`


In [16]:
keyed_df = (
    feature_df
    .withColumn(
        "DateKey",
        F.date_format("FlightDate", "yyyyMMdd").cast("int")
    )
    .withColumn(
        "MarketingAirlineKey",
        F.sha2(
            F.coalesce(
                F.col("Marketing_Airline_Network"),
                F.lit("UNK")
            ),
            256
        )
    )
    .withColumn(
        "OperatingAirlineKey",
        F.sha2(
            F.coalesce(F.col("Operating_Airline"), F.lit("UNK")),
            256
        )
    )
    .withColumn(
        "OriginAirportKey",
        F.sha2(F.coalesce(F.col("Origin"), F.lit("UNK")), 256)
    )
    .withColumn(
        "DestAirportKey",
        F.sha2(F.coalesce(F.col("Dest"), F.lit("UNK")), 256)
    )
    .withColumn(
        "RouteKey",
        F.sha2(F.coalesce(F.col("Route"), F.lit("UNK-UNK")), 256)
    )
)

gold_base_df = keyed_df.persist(StorageLevel.DISK_ONLY)
gold_base_rows = gold_base_df.count()

print("Gold base rows:", gold_base_rows)
print("Gold base storage level:", gold_base_df.storageLevel)


('Gold base rows:', 40910253)
('Gold base storage level:', StorageLevel(True, False, False, False, 1))

# Gold Dimensions

Reliability metrics are stored in dimensions rather than repeated across every fact row.

Reliability score formula:

`100 × (0.70 × OnTimeRate + 0.20 × NonCancellationRate + 0.10 × NonDiversionRate)`


## 16. DIM_DATE

One row per `FlightDate`. Calendar descriptions belong here rather than in the fact table.


In [17]:
dim_date_df = (
    gold_base_df
    .select(
        "DateKey",
        "FlightDate",
        "Year",
        "Quarter",
        "Month",
        "DayofMonth",
        "DayOfWeek",
        "WeekendIndicator",
        "SeasonIndicator",
        "YearMonth"
    )
    .dropDuplicates(["DateKey"])
)

print("DIM_DATE columns:", len(dim_date_df.columns))


('DIM_DATE columns:', 10)

## 17. DIM_AIRLINE

Grain: one row per unique airline code.

Both marketing and operating airline codes are included in the same conformed dimension. Full-period descriptive reliability is calculated only where the airline appears as a marketing carrier.


In [18]:
marketing_airlines_df = gold_base_df.select(
    F.col("MarketingAirlineKey").alias("AirlineKey"),
    F.col("Marketing_Airline_Network").alias("AirlineCode"),
    F.col("MarketingAirlineName").alias("AirlineName"),
    F.col("MarketingAirlineLabel").alias("AirlineLabel")
)

operating_airlines_df = gold_base_df.select(
    F.col("OperatingAirlineKey").alias("AirlineKey"),
    F.col("Operating_Airline").alias("AirlineCode"),
    F.col("OperatingAirlineName").alias("AirlineName"),
    F.col("OperatingAirlineLabel").alias("AirlineLabel")
)

airline_master_df = (
    marketing_airlines_df
    .unionByName(operating_airlines_df)
    .filter(F.col("AirlineCode").isNotNull())
    .groupBy("AirlineKey", "AirlineCode")
    .agg(
        F.first("AirlineName", ignorenulls=True).alias("AirlineName"),
        F.first("AirlineLabel", ignorenulls=True).alias("AirlineLabel")
    )
)

airline_stats_df = (
    gold_base_df
    .groupBy(
        F.col("MarketingAirlineKey").alias("AirlineKey")
    )
    .agg(
        F.count("*").alias("FlightCount"),
        F.avg(
            F.when(
                (F.col("Cancelled") == 0)
                & (F.col("Diverted") == 0)
                & F.col("ArrDelay").isNotNull(),
                F.when(F.col("ArrDelay") <= 15, 1.0).otherwise(0.0)
            )
        ).alias("OnTimeRate"),
        F.avg(
            F.when(F.col("Cancelled") == 1, 1.0).otherwise(0.0)
        ).alias("CancellationRate"),
        F.avg(
            F.when(F.col("Diverted") == 1, 1.0).otherwise(0.0)
        ).alias("DiversionRate")
    )
    .withColumn(
        "ReliabilityScore",
        F.round(
            100 * (
                0.70 * F.coalesce(F.col("OnTimeRate"), F.lit(0.0))
                + 0.20 * (
                    1 - F.coalesce(
                        F.col("CancellationRate"),
                        F.lit(0.0)
                    )
                )
                + 0.10 * (
                    1 - F.coalesce(
                        F.col("DiversionRate"),
                        F.lit(0.0)
                    )
                )
            ),
            2
        )
    )
)

dim_airline_df = (
    airline_master_df
    .join(airline_stats_df, on="AirlineKey", how="left")
    .select(
        "AirlineKey",
        "AirlineCode",
        "AirlineName",
        "AirlineLabel",
        F.coalesce(F.col("FlightCount"), F.lit(0)).alias("FlightCount"),
        F.round("OnTimeRate", 4).alias("OnTimeRate"),
        F.round("CancellationRate", 4).alias("CancellationRate"),
        F.round("DiversionRate", 4).alias("DiversionRate"),
        "ReliabilityScore"
    )
)

print("DIM_AIRLINE columns:", len(dim_airline_df.columns))


('DIM_AIRLINE columns:', 9)

## 18. DIM_AIRPORT

Grain: one row per airport.

The dimension stores descriptive attributes plus separate full-period metrics for:

- Departure reliability when the airport is the origin
- Arrival reliability when the airport is the destination


In [19]:
origin_airports_df = gold_base_df.select(
    F.col("OriginAirportKey").alias("AirportKey"),
    F.col("Origin").alias("AirportCode"),
    F.col("OriginCityName").alias("CityName"),
    F.col("OriginState").alias("StateCode"),
    F.col("OriginStateName").alias("StateName"),
    F.col("OriginRegion").alias("Region")
)

dest_airports_df = gold_base_df.select(
    F.col("DestAirportKey").alias("AirportKey"),
    F.col("Dest").alias("AirportCode"),
    F.col("DestCityName").alias("CityName"),
    F.col("DestState").alias("StateCode"),
    F.col("DestStateName").alias("StateName"),
    F.col("DestRegion").alias("Region")
)

airport_master_df = (
    origin_airports_df
    .unionByName(dest_airports_df)
    .groupBy("AirportKey", "AirportCode")
    .agg(
        F.first("CityName", ignorenulls=True).alias("CityName"),
        F.first("StateCode", ignorenulls=True).alias("StateCode"),
        F.first("StateName", ignorenulls=True).alias("StateName"),
        F.first("Region", ignorenulls=True).alias("Region")
    )
)

departure_stats_df = (
    gold_base_df
    .groupBy(
        F.col("OriginAirportKey").alias("AirportKey")
    )
    .agg(
        F.count("*").alias("DepartureFlightCount"),
        F.avg(
            F.when(
                (F.col("Cancelled") == 0)
                & F.col("DepDelay").isNotNull(),
                F.when(F.col("DepDelay") <= 15, 1.0).otherwise(0.0)
            )
        ).alias("DepartureOnTimeRate"),
        F.avg(
            F.when(F.col("Cancelled") == 1, 1.0).otherwise(0.0)
        ).alias("DepartureCancellationRate"),
        F.avg(
            F.when(F.col("Diverted") == 1, 1.0).otherwise(0.0)
        ).alias("DepartureDiversionRate")
    )
    .withColumn(
        "DepartureReliabilityScore",
        F.round(
            100 * (
                0.70 * F.coalesce(
                    F.col("DepartureOnTimeRate"),
                    F.lit(0.0)
                )
                + 0.20 * (
                    1 - F.coalesce(
                        F.col("DepartureCancellationRate"),
                        F.lit(0.0)
                    )
                )
                + 0.10 * (
                    1 - F.coalesce(
                        F.col("DepartureDiversionRate"),
                        F.lit(0.0)
                    )
                )
            ),
            2
        )
    )
)

arrival_stats_df = (
    gold_base_df
    .groupBy(
        F.col("DestAirportKey").alias("AirportKey")
    )
    .agg(
        F.count("*").alias("ArrivalFlightCount"),
        F.avg(
            F.when(
                (F.col("Cancelled") == 0)
                & (F.col("Diverted") == 0)
                & F.col("ArrDelay").isNotNull(),
                F.when(F.col("ArrDelay") <= 15, 1.0).otherwise(0.0)
            )
        ).alias("ArrivalOnTimeRate"),
        F.round(F.avg("ArrDelay"), 2).alias("AverageArrivalDelay")
    )
    .withColumn(
        "ArrivalReliabilityScore",
        F.round(
            100 * F.coalesce(F.col("ArrivalOnTimeRate"), F.lit(0.0)),
            2
        )
    )
)

dim_airport_df = (
    airport_master_df
    .join(departure_stats_df, on="AirportKey", how="left")
    .join(arrival_stats_df, on="AirportKey", how="left")
    .select(
        "AirportKey",
        "AirportCode",
        "CityName",
        "StateCode",
        "StateName",
        "Region",
        F.coalesce(
            F.col("DepartureFlightCount"),
            F.lit(0)
        ).alias("DepartureFlightCount"),
        F.coalesce(
            F.col("ArrivalFlightCount"),
            F.lit(0)
        ).alias("ArrivalFlightCount"),
        F.round("DepartureOnTimeRate", 4).alias("DepartureOnTimeRate"),
        F.round("ArrivalOnTimeRate", 4).alias("ArrivalOnTimeRate"),
        F.round(
            "DepartureCancellationRate",
            4
        ).alias("DepartureCancellationRate"),
        F.round(
            "DepartureDiversionRate",
            4
        ).alias("DepartureDiversionRate"),
        "AverageArrivalDelay",
        "DepartureReliabilityScore",
        "ArrivalReliabilityScore"
    )
)

print("DIM_AIRPORT columns:", len(dim_airport_df.columns))


('DIM_AIRPORT columns:', 15)

## 19. DIM_ROUTE

One row per directional airport pair. `ATL-JFK` and `JFK-ATL` are different routes.


In [20]:
dim_route_df = (
    gold_base_df
    .groupBy(
        "RouteKey",
        "Route",
        "Origin",
        "Dest",
        "StatePair"
    )
    .agg(
        F.count("*").alias("FlightCount"),
        F.round(F.avg("ArrDelay"), 2).alias("AverageDelay"),
        F.avg(
            F.when(
                (F.col("Cancelled") == 0)
                & (F.col("Diverted") == 0)
                & F.col("ArrDelay").isNotNull(),
                F.when(F.col("ArrDelay") <= 15, 1.0).otherwise(0.0)
            )
        ).alias("OnTimeRate"),
        F.avg(F.when(F.col("Cancelled") == 1, 1.0).otherwise(0.0)).alias("CancellationRate"),
        F.avg(F.when(F.col("Diverted") == 1, 1.0).otherwise(0.0)).alias("DiversionRate")
    )
    .withColumn(
        "ReliabilityScore",
        F.round(
            100 * (
                0.70 * F.coalesce(F.col("OnTimeRate"), F.lit(0.0))
                + 0.20 * (1 - F.coalesce(F.col("CancellationRate"), F.lit(0.0)))
                + 0.10 * (1 - F.coalesce(F.col("DiversionRate"), F.lit(0.0)))
            ),
            2
        )
    )
    .select(
        "RouteKey",
        "Route",
        "Origin",
        "Dest",
        "StatePair",
        "FlightCount",
        "AverageDelay",
        F.round("OnTimeRate", 4).alias("OnTimeRate"),
        F.round("CancellationRate", 4).alias("CancellationRate"),
        F.round("DiversionRate", 4).alias("DiversionRate"),
        "ReliabilityScore"
    )
)

print("DIM_ROUTE columns:", len(dim_route_df.columns))


('DIM_ROUTE columns:', 11)

# Gold Fact Table

## 20. FACT_FLIGHTS

Grain: one row per flight.

Reliability values are intentionally excluded from the Fact table. `Year` and `Month` are retained as physical partition columns.


In [21]:
fact_flights_columns = [
    # Business and dimension keys
    "FlightKey",
    "DateKey",
    "MarketingAirlineKey",
    "OperatingAirlineKey",
    "OriginAirportKey",
    "DestAirportKey",
    "RouteKey",

    # Physical partition columns and flight identifier
    "Year",
    "Month",
    "FlightDate",
    "Flight_Number_Marketing_Airline",
    "Operated_or_Branded_Code_Share_Partners",

    # Scheduled time features
    "CRSDepTime",
    "CRSArrTime",
    "ScheduledElapsedTimeMinutes",
    "DepartureHour",
    "ArrivalHour",
    "DeparturePeriod",
    "ArrivalPeriod",
    "PeakHourIndicator",
    "WeekendIndicator",
    "SeasonIndicator",

    # Core measures
    "Distance",
    "AirTime",
    "TaxiOut",
    "TaxiIn",
    "DepDelay",
    "ArrDelay",
    "DepDel15",
    "ArrDel15",

    # Delay availability and exception flags
    "HasDepDelay",
    "HasArrDelay",
    "ArrDelayNullExceptionFlag",

    # Flight outcome
    "Cancelled",
    "Diverted",
    "CompletedFlightFlag",
    "FlightStatus",

    # Categories and operational features
    "DelayCategory",
    "DeparturePerformanceType",
    "ArrivalPerformanceType",
    "DistanceCategory",
    "FlightDurationCategory",
    "FlightTimeCompleteFlag",
    "TotalFlightTimeMinutes",
    "AverageFlightSpeedMph",

    # Delay-cause detail
    "CarrierDelay",
    "WeatherDelay",
    "NASDelay",
    "SecurityDelay",
    "LateAircraftDelay",
    "PrimaryDelayCause",
    "TotalRecordedCauseDelay",
    "DelayCauseFlag",

    # Route/code-share flags
    "CodeshareFlag",
    "IntraStateRouteFlag"
]

fact_flights_df = gold_base_df.select(*fact_flights_columns)

print("FACT_FLIGHTS columns:", len(fact_flights_df.columns))
fact_flights_df.printSchema()


('FACT_FLIGHTS columns:', 55)
root
 |-- FlightKey: string (nullable = false)
 |-- DateKey: integer (nullable = true)
 |-- MarketingAirlineKey: string (nullable = true)
 |-- OperatingAirlineKey: string (nullable = true)
 |-- OriginAirportKey: string (nullable = true)
 |-- DestAirportKey: string (nullable = true)
 |-- RouteKey: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- FlightDate: date (nullable = true)
 |-- Flight_Number_Marketing_Airline: integer (nullable = true)
 |-- Operated_or_Branded_Code_Share_Partners: string (nullable = true)
 |-- CRSDepTime: integer (nullable = true)
 |-- CRSArrTime: integer (nullable = true)
 |-- ScheduledElapsedTimeMinutes: integer (nullable = true)
 |-- DepartureHour: integer (nullable = true)
 |-- ArrivalHour: integer (nullable = true)
 |-- DeparturePeriod: string (nullable = false)
 |-- ArrivalPeriod: string (nullable = false)
 |-- PeakHourIndicator: integer (nullable = false)
 |-- WeekendIndi

# Machine-Learning Dataset

The pre-departure ML dataset predicts `ArrDel15`.

It excludes actual operational and post-flight fields such as `DepDelay`, `ArrDelay`, `AirTime`, taxi times, actual delay causes, flight status, and actual flight speed.


## 21. ML_DATASET for pre-departure arrival-delay prediction

### Time split

- Train: flights on or before `TRAIN_END_DATE`
- Validation: `VALIDATION_YEAR`
- Test: `TEST_YEAR`

### Leakage prevention

Airline, origin-airport, destination-airport, and route reliability features are calculated from training-history records only. They are then joined to train, validation, and test flights as frozen historical features.


In [22]:
training_history_df = gold_base_df.filter(
    F.col("FlightDate") <= F.to_date(F.lit(TRAIN_END_DATE))
)

# Training-history airline reliability
train_airline_scores_df = (
    training_history_df
    .groupBy(
        F.col("MarketingAirlineKey").alias("AirlineKey")
    )
    .agg(
        F.avg(
            F.when(
                (F.col("Cancelled") == 0)
                & (F.col("Diverted") == 0)
                & F.col("ArrDelay").isNotNull(),
                F.when(F.col("ArrDelay") <= 15, 1.0).otherwise(0.0)
            )
        ).alias("OnTimeRate"),
        F.avg(
            F.when(F.col("Cancelled") == 1, 1.0).otherwise(0.0)
        ).alias("CancellationRate"),
        F.avg(
            F.when(F.col("Diverted") == 1, 1.0).otherwise(0.0)
        ).alias("DiversionRate")
    )
    .withColumn(
        "AirlineReliabilityScore",
        F.round(
            100 * (
                0.70 * F.coalesce(F.col("OnTimeRate"), F.lit(0.0))
                + 0.20 * (
                    1 - F.coalesce(
                        F.col("CancellationRate"),
                        F.lit(0.0)
                    )
                )
                + 0.10 * (
                    1 - F.coalesce(
                        F.col("DiversionRate"),
                        F.lit(0.0)
                    )
                )
            ),
            2
        )
    )
    .select("AirlineKey", "AirlineReliabilityScore")
)

# Training-history origin-airport departure reliability
train_origin_scores_df = (
    training_history_df
    .groupBy(
        F.col("OriginAirportKey").alias("AirportKey")
    )
    .agg(
        F.avg(
            F.when(
                (F.col("Cancelled") == 0)
                & F.col("DepDelay").isNotNull(),
                F.when(F.col("DepDelay") <= 15, 1.0).otherwise(0.0)
            )
        ).alias("OnTimeRate"),
        F.avg(
            F.when(F.col("Cancelled") == 1, 1.0).otherwise(0.0)
        ).alias("CancellationRate"),
        F.avg(
            F.when(F.col("Diverted") == 1, 1.0).otherwise(0.0)
        ).alias("DiversionRate")
    )
    .withColumn(
        "OriginAirportReliabilityScore",
        F.round(
            100 * (
                0.70 * F.coalesce(F.col("OnTimeRate"), F.lit(0.0))
                + 0.20 * (
                    1 - F.coalesce(
                        F.col("CancellationRate"),
                        F.lit(0.0)
                    )
                )
                + 0.10 * (
                    1 - F.coalesce(
                        F.col("DiversionRate"),
                        F.lit(0.0)
                    )
                )
            ),
            2
        )
    )
    .select("AirportKey", "OriginAirportReliabilityScore")
)

# Training-history destination-airport arrival reliability
train_dest_scores_df = (
    training_history_df
    .groupBy(
        F.col("DestAirportKey").alias("AirportKey")
    )
    .agg(
        F.avg(
            F.when(
                (F.col("Cancelled") == 0)
                & (F.col("Diverted") == 0)
                & F.col("ArrDelay").isNotNull(),
                F.when(F.col("ArrDelay") <= 15, 1.0).otherwise(0.0)
            )
        ).alias("ArrivalOnTimeRate")
    )
    .withColumn(
        "DestAirportReliabilityScore",
        F.round(
            100 * F.coalesce(F.col("ArrivalOnTimeRate"), F.lit(0.0)),
            2
        )
    )
    .select("AirportKey", "DestAirportReliabilityScore")
)

# Training-history route reliability
train_route_scores_df = (
    training_history_df
    .groupBy("RouteKey")
    .agg(
        F.avg(
            F.when(
                (F.col("Cancelled") == 0)
                & (F.col("Diverted") == 0)
                & F.col("ArrDelay").isNotNull(),
                F.when(F.col("ArrDelay") <= 15, 1.0).otherwise(0.0)
            )
        ).alias("OnTimeRate"),
        F.avg(
            F.when(F.col("Cancelled") == 1, 1.0).otherwise(0.0)
        ).alias("CancellationRate"),
        F.avg(
            F.when(F.col("Diverted") == 1, 1.0).otherwise(0.0)
        ).alias("DiversionRate")
    )
    .withColumn(
        "RouteReliabilityScore",
        F.round(
            100 * (
                0.70 * F.coalesce(F.col("OnTimeRate"), F.lit(0.0))
                + 0.20 * (
                    1 - F.coalesce(
                        F.col("CancellationRate"),
                        F.lit(0.0)
                    )
                )
                + 0.10 * (
                    1 - F.coalesce(
                        F.col("DiversionRate"),
                        F.lit(0.0)
                    )
                )
            ),
            2
        )
    )
    .select("RouteKey", "RouteReliabilityScore")
)

ml_base_df = (
    gold_base_df
    .filter(
        (F.col("Cancelled") == 0)
        & (F.col("Diverted") == 0)
        & F.col("ArrDel15").isNotNull()
    )
    .select(
        "FlightKey",
        "FlightDate",
        "Year",
        "Quarter",
        "Month",
        "DayofMonth",
        "DayOfWeek",
        "DepartureHour",
        "ArrivalHour",
        "DeparturePeriod",
        "ArrivalPeriod",
        "PeakHourIndicator",
        "WeekendIndicator",
        "SeasonIndicator",
        "MarketingAirlineKey",
        "OperatingAirlineKey",
        "OriginAirportKey",
        "DestAirportKey",
        "RouteKey",
        "Distance",
        "ScheduledElapsedTimeMinutes",
        "DistanceCategory",
        "CodeshareFlag",
        "IntraStateRouteFlag",
        "ArrDel15"
    )
)

ml_dataset_df = (
    ml_base_df
    .join(
        train_airline_scores_df.withColumnRenamed(
            "AirlineKey",
            "MarketingAirlineKey"
        ),
        on="MarketingAirlineKey",
        how="left"
    )
    .join(
        train_origin_scores_df
        .withColumnRenamed("AirportKey", "OriginAirportKey"),
        on="OriginAirportKey",
        how="left"
    )
    .join(
        train_dest_scores_df
        .withColumnRenamed("AirportKey", "DestAirportKey"),
        on="DestAirportKey",
        how="left"
    )
    .join(train_route_scores_df, on="RouteKey", how="left")
    .withColumn(
        "DatasetSplit",
        F.when(
            F.col("FlightDate") <= F.to_date(F.lit(TRAIN_END_DATE)),
            "Train"
        )
        .when(F.col("Year") == VALIDATION_YEAR, "Validation")
        .when(F.col("Year") == TEST_YEAR, "Test")
        .otherwise("OutsideConfiguredSplit")
    )
    .withColumn(
        "ReliabilityFeatureScope",
        F.lit("TRAINING_HISTORY_ONLY")
    )
)

print("ML_DATASET columns:", len(ml_dataset_df.columns))
print("ML target: ArrDel15")
ml_dataset_df.groupBy("DatasetSplit").count().show()


('ML_DATASET columns:', 31)
ML target: ArrDel15
+------------+--------+
|DatasetSplit|   count|
+------------+--------+
|  Validation| 7425229|
|        Test| 7597495|
|       Train|24872650|
+------------+--------+

## 22. Post-flight fields intentionally excluded from the pre-departure ML dataset

These columns are useful for operational analysis but must not be used when predicting before departure because they are unknown at prediction time.


In [23]:
post_flight_optional_columns = [
    "DepDelay",
    "ArrDelay",
    "AirTime",
    "TaxiOut",
    "TaxiIn",
    "AverageFlightSpeedMph",
    "CarrierDelay",
    "WeatherDelay",
    "NASDelay",
    "SecurityDelay",
    "LateAircraftDelay"
]

# Example only; not written by default:
# ml_post_flight_df = ml_dataset_df.join(
#     gold_base_df.select("FlightKey", *post_flight_optional_columns),
#     on="FlightKey",
#     how="left"
# )


# Minimal Output Integrity Checks

Detailed data-quality reporting is generated separately in `Gold_Validation_Report.ipynb`.

This transformation notebook keeps only fail-fast checks that prevent invalid Gold output.


## 23. Required-key and row-count checks

In [24]:
fact_count = fact_flights_df.count()

if fact_count != gold_base_rows:
    raise ValueError(
        "FACT_FLIGHTS row count differs from the Silver-to-Gold base"
    )

required_key_nulls = fact_flights_df.select(
    F.sum(
        F.when(F.col("FlightKey").isNull(), 1).otherwise(0)
    ).alias("MissingFlightKey"),
    F.sum(
        F.when(F.col("DateKey").isNull(), 1).otherwise(0)
    ).alias("MissingDateKey"),
    F.sum(
        F.when(F.col("MarketingAirlineKey").isNull(), 1).otherwise(0)
    ).alias("MissingMarketingAirlineKey"),
    F.sum(
        F.when(F.col("OperatingAirlineKey").isNull(), 1).otherwise(0)
    ).alias("MissingOperatingAirlineKey"),
    F.sum(
        F.when(F.col("OriginAirportKey").isNull(), 1).otherwise(0)
    ).alias("MissingOriginAirportKey"),
    F.sum(
        F.when(F.col("DestAirportKey").isNull(), 1).otherwise(0)
    ).alias("MissingDestAirportKey"),
    F.sum(
        F.when(F.col("RouteKey").isNull(), 1).otherwise(0)
    ).alias("MissingRouteKey")
)

required_key_nulls.show(truncate=False)
print("FACT_FLIGHTS row count preserved:", fact_count == gold_base_rows)


+----------------+--------------+--------------------------+--------------------------+-----------------------+---------------------+---------------+
|MissingFlightKey|MissingDateKey|MissingMarketingAirlineKey|MissingOperatingAirlineKey|MissingOriginAirportKey|MissingDestAirportKey|MissingRouteKey|
+----------------+--------------+--------------------------+--------------------------+-----------------------+---------------------+---------------+
|0               |0             |0                         |0                         |0                      |0                    |0              |
+----------------+--------------+--------------------------+--------------------------+-----------------------+---------------------+---------------+

('FACT_FLIGHTS row count preserved:', True)

## 24. Lightweight previews

These previews inspect each output independently.

They deliberately avoid constructing a single preview containing Fact rows plus all reliability dimensions, which previously triggered expensive broadcast/join execution.


In [25]:
fact_flights_df.select(
    "FlightKey",
    "FlightDate",
    "Year",
    "Month",
    "MarketingAirlineKey",
    "OperatingAirlineKey",
    "OriginAirportKey",
    "DestAirportKey",
    "RouteKey",
    "DeparturePeriod",
    "DelayCategory",
    "FlightStatus"
).limit(5).show(truncate=False)

dim_airline_df.limit(10).show(truncate=False)

dim_airport_df.select(
    "AirportKey",
    "AirportCode",
    "CityName",
    "StateCode",
    "DepartureReliabilityScore",
    "ArrivalReliabilityScore"
).limit(10).show(truncate=False)

dim_date_df.orderBy("FlightDate").limit(10).show(truncate=False)

dim_route_df.select(
    "RouteKey",
    "Route",
    "FlightCount",
    "AverageDelay",
    "ReliabilityScore"
).orderBy(F.desc("FlightCount")).limit(10).show(truncate=False)

# Preview the ML base without forcing all historical reliability joins.
ml_base_df.select(
    "FlightDate",
    "Year",
    "Month",
    "DepartureHour",
    "ArrivalHour",
    "MarketingAirlineKey",
    "OriginAirportKey",
    "DestAirportKey",
    "RouteKey",
    "Distance",
    "ArrDel15"
).limit(5).show(truncate=False)


+-------------------------------+----------+----+-----+----------------------------------------------------------------+----------------------------------------------------------------+----------------------------------------------------------------+----------------------------------------------------------------+----------------------------------------------------------------+---------------+-------------+------------+
|FlightKey                      |FlightDate|Year|Month|MarketingAirlineKey                                             |OperatingAirlineKey                                             |OriginAirportKey                                                |DestAirportKey                                                  |RouteKey                                                        |DeparturePeriod|DelayCategory|FlightStatus|
+-------------------------------+----------+----+-----+----------------------------------------------------------------+--------------------------------

# Write Gold Outputs

## 25. Write all six outputs

`FACT_FLIGHTS` and `ML_DATASET` are physically partitioned by `Year` and `Month`.


In [26]:
if WRITE_OUTPUT:
    print("Writing DIM_DATE...")
    dim_date_df.write.mode(OUTPUT_MODE).parquet(DIM_DATE_PATH)

    print("Writing DIM_AIRLINE...")
    dim_airline_df.write.mode(OUTPUT_MODE).parquet(DIM_AIRLINE_PATH)

    print("Writing DIM_AIRPORT...")
    dim_airport_df.write.mode(OUTPUT_MODE).parquet(DIM_AIRPORT_PATH)

    print("Writing DIM_ROUTE...")
    dim_route_df.write.mode(OUTPUT_MODE).parquet(DIM_ROUTE_PATH)

    print("Writing FACT_FLIGHTS partitioned by Year and Month...")
    (
        fact_flights_df
        .write
        .mode(OUTPUT_MODE)
        .partitionBy("Year", "Month")
        .parquet(FACT_FLIGHTS_PATH)
    )

    print("Writing ML_DATASET partitioned by Year and Month...")
    (
        ml_dataset_df
        .write
        .mode(OUTPUT_MODE)
        .partitionBy("Year", "Month")
        .parquet(ML_DATASET_PATH)
    )

    print("All six Gold outputs were written successfully.")
else:
    print(
        "Write skipped. Set WRITE_OUTPUT = True only after "
        "the transformations and lightweight checks complete."
    )


Writing DIM_DATE...
Writing DIM_AIRLINE...
Writing DIM_AIRPORT...
Writing DIM_ROUTE...
Writing FACT_FLIGHTS partitioned by Year and Month...
Writing ML_DATASET partitioned by Year and Month...
All six Gold outputs were written successfully.

## 26. Release persisted data

In [27]:
gold_base_df.unpersist(blocking=True)
print("Released persisted Gold base after all output operations.")


Released persisted Gold base after all output operations.

# Conclusion

The Silver-to-Gold transformation pipeline was successfully executed to convert the cleaned Silver dataset into a business-ready Gold layer following a Star Schema architecture for analytics and machine learning.

## Objectives Achieved

- Loaded the cleaned Silver dataset from Amazon S3.
- Performed feature engineering to derive business-relevant attributes.
- Generated surrogate and business keys for dimensional modelling.
- Created a centralized Gold Base dataset for downstream processing.
- Built dimension tables for Airline, Airport, Date, and Route.
- Constructed the Fact Flights table using foreign keys to the dimension tables.
- Generated a machine learning dataset with historical reliability features while preventing target leakage through time-based feature generation.
- Applied Train (2020–2023), Validation (2024), and Test (2025) data splits for future predictive modelling.
- Stored all Gold datasets in partitioned Parquet format on Amazon S3 for efficient querying and analytics.

## Gold Layer Outputs

The following datasets were created:

- **FACT_FLIGHTS**
- **DIM_AIRLINE**
- **DIM_AIRPORT**
- **DIM_DATE**
- **DIM_ROUTE**
- **ML_DATASET**

## Feature Engineering Completed

Business-oriented features created include:

- Departure Hour
- Arrival Hour
- Departure Period
- Arrival Period
- Peak Hour Indicator
- Weekend Indicator
- Season Indicator
- Delay Category
- Flight Duration Category
- Distance Category
- Flight Status
- Route Key
- Flight Key
- Codeshare Flag
- Intra-State Route Flag
- Total Flight Time
- Airline Reliability Score
- Origin Airport Reliability Score
- Destination Airport Reliability Score
- Route Reliability Score

## Machine Learning Dataset

The ML dataset was prepared specifically for predicting **Arrival Delay (ArrDel15)**.

- Historical reliability scores computed only from past data
- Dataset split labels (Train, Validation, Test)

This ensures there is no data leakage while training future machine learning models.

## Final Architecture

```
Bronze
        │
        ▼
Silver
(Cleaned & Standardized Dataset)
        │
        ▼
Gold Base
        │
        ├──────────────► FACT_FLIGHTS
        │
        ├──────────────► DIM_AIRLINE
        │
        ├──────────────► DIM_AIRPORT
        │
        ├──────────────► DIM_DATE
        │
        ├──────────────► DIM_ROUTE
        │
        └──────────────► ML_DATASET
```

## Outcome

The Gold layer provides a scalable and analytics-ready data model that supports:

- Business Intelligence dashboards
- KPI reporting
- Historical trend analysis
- Star Schema querying
- Predictive Machine Learning

